# Qwen3 Approach Steering & Token Trace Demo

検証済みapproach方向をスライダーで観察する研究デモです。source 18で方向を作り、target 20へ介入します。semantic方向と5本の直交random方向を同じalphaで比較できます。観測のみ・最初のtokenだけ・全token継続介入の3モードを比較できます。`enable thinking`を外すと、chat templateと方向をno-thinking条件で作り直します。この条件と継続介入は探索条件で、検証済み主結果ではありません。安全機構・権限制御・人格や感情の実装ではありません。**無害なpromptだけで使用してください。**

In [ ]:
!nvidia-smi
!pip -q uninstall -y gradio gradio_client
!pip -q install "transformers==4.53.2" accelerate ipywidgets matplotlib pandas

`colab_neurostate_3axis.py` と `colab_approach_steering_demo.py` を同時にアップロードしてください。

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
for stem in ('colab_neurostate_3axis','colab_approach_steering_demo'):
    candidates=[name for name in uploaded if name.startswith(stem) and name.endswith('.py')]
    assert candidates,f'{stem}.py がありません'
    Path(stem+'.py').write_bytes(uploaded[candidates[-1]])
print('files ready')

## モデルとapproach方向を準備

初回だけモデル読み込みと10 promptからの方向構築を行います。

In [ ]:
from colab_approach_steering_demo import ApproachSteeringDemo
demo=ApproachSteeringDemo()
print('demo ready')

## スライダーとtoken traceで観察

alphaが正なら選択した方向の正側、負なら負側です。表示するlogit contrastは `proceed語群 − hesitate語群`。まずsemanticとrandom #1〜#5を同じprompt・alphaで比較してください。observeは観測のみ、boundaryは最初のtokenだけ、continuousは生成中の全tokenへ介入します。`enable thinking`がオンの条件が既存結果に対応します。オフでは初回のみ別方向bankの構築に時間がかかります。`extended alpha ±4`は検証範囲外の探索設定です。反復・崩壊・順位分岐そのものを観察し、結果を既存holdoutと混同しないでください。

In [ ]:
import ipywidgets as widgets
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display,clear_output
prompt=widgets.Textarea(value='Write one short sentence about checking a calendar.',description='Prompt:',layout=widgets.Layout(width='90%',height='90px'))
alpha=widgets.FloatSlider(value=0.0,min=-2.0,max=2.0,step=0.25,description='alpha')
direction=widgets.RadioButtons(options=['semantic','random #1','random #2','random #3','random #4','random #5'],value='semantic',description='direction')
thinking=widgets.Checkbox(value=True,description='enable thinking (validated default)',indent=False)
extended=widgets.Checkbox(value=False,description='extended alpha ±4 (exploratory)',indent=False)
mode=widgets.Dropdown(options=[('Observe only','observe'),('Boundary: first token','boundary'),('Continuous: every token (exploratory)','continuous')],value='boundary',description='mode')
tokens=widgets.IntSlider(value=40,min=8,max=96,step=8,description='tokens')
button=widgets.Button(description='Run trace',button_style='primary')
out=widgets.Output()
def set_alpha_range(change=None):
    limit=4.0 if extended.value else 2.0
    alpha.min=-limit;alpha.max=limit
    alpha.value=max(-limit,min(limit,alpha.value))
extended.observe(set_alpha_range,names='value')
def run(_):
    with out:
        clear_output(wait=True)
        result=demo.trace(prompt.value,alpha.value,mode.value,tokens.value,direction_name=direction.value,enable_thinking=thinking.value,allow_extended_alpha=extended.value)
        rows=[{'step':s.step,'raw_token':s.raw_token,'token':repr(s.token),'cumulative':s.cumulative_text,'baseline_top':repr(s.baseline_token),'choice_top':repr(s.token),'argmax_changed':s.argmax_changed,'applied':s.intervention_applied,'baseline_margin':s.baseline_margin,'steered_margin':s.steered_margin,'baseline':s.baseline_contrast,'steered':s.steered_contrast,'delta':s.delta} for s in result['steps']]
        frame=pd.DataFrame(rows)
        condition='thinking' if result['enable_thinking'] else 'no-thinking exploratory'
        print(f"condition={condition} direction={result['direction_name']} mode={result['mode']} alpha={result['alpha']:+.2f}")
        print('\nGenerated:')
        print(result['generated'])
        changed=frame.index[frame['argmax_changed']].tolist()
        print('\nFirst argmax divergence:',changed[0] if changed else 'none')
        display(frame)
        plt.figure(figsize=(10,4))
        plt.plot(frame['step'],frame['baseline'],label='pre-intervention contrast',marker='.')
        plt.plot(frame['step'],frame['steered'],label='choice contrast',marker='.')
        plt.axhline(0,color='gray',linewidth=1)
        plt.xlabel('generated token step');plt.ylabel('proceed - hesitate logit')
        plt.title('Internal contrast vs emitted token sequence')
        plt.legend();plt.show()
button.on_click(run)
display(prompt,alpha,direction,thinking,extended,mode,tokens,button,out)

最初にthinkingをオンのまま、同じprompt・alphaでsemantic / random #1〜#5を比較し、その後observe / boundary / continuousを比較してください。no-thinkingではchat templateとsemantic/random方向をその条件専用に再構築します。既存holdoutとの直接比較はせず探索結果として扱います。表の`raw_token`と`cumulative`で日本語のbyte断片と復元過程を、`baseline_top` / `choice_top` / marginで順位境界を確認できます。`argmax_changed`が最初にTrueになるstepが出力分岐点です。randomでも文章が変わることがあるため、1回の文章差だけを意味方向の証拠にはできません。continuousとextended alphaは検証済みの境界介入とは別条件です。削除・送信・購入・認証などの実操作へ接続しないでください。